# A/A Test: City Lookalike (MAVERICK + CausalImpact)

Validates the City Lookalike synthetic control by running BSTS against
historical non-campaign periods. Expected result: **zero uplift**.

For each treatment city:
1. Find control cities via KNN on city features (MAVERICK approach)
2. Filter by weekly order correlation >= threshold
3. Run CausalImpact with individual control time series
4. Write results to shared BQ table

---
## 1. Config & Authentication

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import time, gc, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

!pip install -q tfcausalimpact
from causalimpact import CausalImpact
print('CausalImpact installed')

# Mount Google Drive (one-time auth prompt)
from google.colab import drive
drive.mount('/content/drive')

# Point Python to the config directory on Drive
# Upload aa_config.py + campaign_data.csv to this folder ONCE
import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)  # so aa_config can find campaign_data.csv

import aa_config as cfg

client = bigquery.Client(project=cfg.PROJECT_ID)
print(f'Connected to {cfg.PROJECT_ID}')
cities = {c: cfg.CITY_TREATMENT_CANDIDATES.get(c,[]) for c in cfg.COUNTRIES}
total = sum(len(v)*len(cfg.TIME_WINDOWS.get(c,[])) for c,v in cities.items())
print(f'Treatment cities: {cities}')
print(f'Total runs: {total}')

---
## 2. City Features (MAVERICK)

In [ ]:
def get_city_features(client, country, ref_date):
    query = f"""
    WITH active AS (
        SELECT dr.city, SUM(fo.nroforders) AS total_ov,
            COUNT(DISTINCT fo.customerid) AS unique_customers
        FROM `just-data-warehouse.dwh.fact_order` AS fo
        INNER JOIN `just-data-warehouse.dwh.dim_restaurant` AS dr ON fo.restaurantid=dr.restaurantid
        WHERE dr.country='{country}'
          AND DATE(fo.orderdatetime) >= DATE_SUB(DATE('{ref_date}'), INTERVAL 90 DAY)
          AND DATE(fo.orderdatetime) < DATE('{ref_date}')
        GROUP BY 1
    ),
    population AS (
        SELECT dz.city, SUM(pop_addressable) AS pop_15plus
        FROM `just-data-warehouse.cma_mbr_population.mbr_population` AS pop
        INNER JOIN `just-data-warehouse.dwh.dim_zipcode` AS dz ON pop.tableau_map_zipcode=dz.zipcode
        WHERE pop.country='{country}' AND dz.country='{country}'
          AND year=(SELECT MAX(year) FROM `just-data-warehouse.cma_mbr_population.mbr_population`)
        GROUP BY 1
    ),
    supply AS (
        SELECT dr.city, COUNT(dr.restaurantid) AS online_partners,
            SUM(CASE WHEN dr.chain='(Not available)' THEN 1 ELSE 0 END) AS smb_partners,
            SUM(CASE WHEN dr.chain!='(Not available)' THEN 1 ELSE 0 END) AS chain_partners
        FROM `just-data-warehouse.dwh.dim_restaurant` AS dr
        WHERE dr.country='{country}'
          AND dr.onlinestatus IN ('Online (temporarily closed)','Online (with Menu)')
        GROUP BY 1
    )
    SELECT a.city, COALESCE(p.pop_15plus,0) AS pop_15plus,
        a.total_ov, a.unique_customers,
        COALESCE(s.online_partners,0) AS online_partners,
        COALESCE(s.smb_partners,0) AS smb_partners,
        COALESCE(s.chain_partners,0) AS chain_partners,
        SAFE_DIVIDE(a.total_ov, p.pop_15plus) AS ov_per_capita
    FROM active a
    LEFT JOIN population p ON a.city=p.city
    LEFT JOIN supply s ON a.city=s.city
    WHERE a.total_ov > 0
    """
    return client.query(query).to_dataframe()

print('City features function loaded.')

---
## 3. Control City Selection & Correlation Filter

In [ ]:
def find_control_cities(df_cities, treatment_city, n_neighbors=25):
    if treatment_city not in df_cities['city'].values:
        print(f'  WARNING: {treatment_city} not found')
        return []
    num_cols = df_cities.select_dtypes(include=['number']).columns.tolist()
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df_cities[num_cols].fillna(0))
    target_idx = df_cities[df_cities['city']==treatment_city].index[0]
    target_pos = list(df_cities.index).index(target_idx)
    n = min(n_neighbors+1, len(df_cities))
    knn = NearestNeighbors(n_neighbors=n, algorithm='auto')
    knn.fit(scaled)
    _, indices = knn.kneighbors(scaled[target_pos].reshape(1,-1))
    return [df_cities.iloc[i]['city'] for i in indices.flatten()
            if df_cities.iloc[i]['city'] != treatment_city][:n_neighbors]


def get_daily_orders(client, country, cities, start, end, segment=None):
    city_list = ','.join([f"'{c}'" for c in cities])
    seg_join, seg_filter = '', ''
    if segment:
        seg_join = """
        LEFT JOIN `just-data-warehouse.core_dwh.dim_unique_customer_history` AS ch
            ON fo.customerid=ch.customer_id
            AND DATE_SUB(ch.snapshot_date, INTERVAL 1 DAY)=DATE(fo.orderdatetime)
        LEFT JOIN `just-data-warehouse.dwh.fact_segmentation_scv_key` AS s
            ON ch.scv_key=s.scv_key
            AND DATE_SUB(DATE(fo.orderdatetime), INTERVAL 1 DAY)=s.snapshot_date"""
        seg_filter = f"AND COALESCE(s.base_segment,'unknown')='{segment}'"
    query = f"""
    SELECT DATE(fo.orderdatetime) AS orderdate, dr.city, SUM(fo.nroforders) AS totalorders
    FROM `just-data-warehouse.dwh.fact_order` AS fo
    INNER JOIN `just-data-warehouse.dwh.dim_restaurant` AS dr ON fo.restaurantid=dr.restaurantid
    {seg_join}
    WHERE dr.country='{country}' AND dr.city IN ({city_list})
      AND DATE(fo.orderdatetime) >= DATE('{start}')
      AND DATE(fo.orderdatetime) <= DATE('{end}')
      {seg_filter}
    GROUP BY 1,2 ORDER BY 1,2
    """
    return client.query(query).to_dataframe()


def apply_correlation_filter(df_orders, treatment_city, controls, threshold):
    all_cities = [treatment_city] + controls
    pivot = df_orders[df_orders['city'].isin(all_cities)].pivot_table(
        index='orderdate', columns='city', values='totalorders').fillna(0)
    pivot.index = pd.to_datetime(pivot.index)
    weekly = pivot.resample('W-MON').sum()
    if treatment_city not in weekly.columns:
        return []
    corr = weekly.corr()[treatment_city]
    passed = corr[(corr >= threshold) & (corr.index != treatment_city)]
    print(f'  Correlation: {len(passed)}/{len(controls)} pass (>={threshold})')
    return passed.index.tolist()

print('Control selection functions loaded.')

---
## 4. CausalImpact Runner

In [ ]:
def run_causal_impact(df_orders, treatment_city, controls, pre_period, post_period):
    all_cities = [treatment_city] + controls
    pivot = df_orders[df_orders['city'].isin(all_cities)].pivot_table(
        index='orderdate', columns='city', values='totalorders').fillna(0)
    pivot.index = pd.to_datetime(pivot.index)
    cols = [treatment_city] + [c for c in pivot.columns if c != treatment_city]
    pivot = pivot[cols]
    pre = [pd.to_datetime(d) for d in pre_period]
    post = [pd.to_datetime(d) for d in post_period]
    try:
        ci = CausalImpact(pivot, pre, post, model_args={'fit_method':'hmc'})
        summary = ci.summary_data
        actual = summary.loc['actual','cumulative']
        predicted = summary.loc['predicted','cumulative']
        uplift = (actual/predicted - 1) if predicted > 0 else np.nan
        return {'actual_orders': actual, 'predicted_orders': predicted,
                'uplift': uplift, 'ci_p_value': ci.p_value if hasattr(ci,'p_value') else np.nan}
    except Exception as e:
        print(f'  CausalImpact error: {e}')
        return None

print('CausalImpact runner loaded.')

---
## 5. Write Results to BQ

In [ ]:
def write_city_result(client, row_dict):
    df = pd.DataFrame([row_dict])
    schema = [
        bigquery.SchemaField('technique', 'STRING'),
        bigquery.SchemaField('country', 'STRING'),
        bigquery.SchemaField('window_start', 'STRING'),
        bigquery.SchemaField('window_end', 'STRING'),
        bigquery.SchemaField('seed', 'INTEGER'),
        bigquery.SchemaField('treatment_city', 'STRING'),
        bigquery.SchemaField('base_segment', 'STRING'),
        bigquery.SchemaField('n_units', 'INTEGER'),
        bigquery.SchemaField('n_controls', 'INTEGER'),
        bigquery.SchemaField('exact_match_pct', 'FLOAT'),
        bigquery.SchemaField('pre_period_uplift', 'FLOAT'),
        bigquery.SchemaField('campaign_period_uplift', 'FLOAT'),
        bigquery.SchemaField('post_period_uplift', 'FLOAT'),
        bigquery.SchemaField('campaign_incr_orders', 'FLOAT'),
        bigquery.SchemaField('post_incr_orders', 'FLOAT'),
        bigquery.SchemaField('run_timestamp', 'TIMESTAMP'),
    ]
    job = client.load_table_from_dataframe(
        df[[s.name for s in schema]], cfg.AA_RESULTS_TABLE,
        job_config=bigquery.LoadJobConfig(schema=schema, write_disposition='WRITE_APPEND'))
    job.result()

print('Write function loaded.')

---
## 6. Run A/A Test

In [ ]:
start_time = time.time()
run = 0

for country in cfg.COUNTRIES:
    candidates = cfg.CITY_TREATMENT_CANDIDATES.get(country, [])
    if not candidates:
        print(f'No city candidates for {country}')
        continue

    for window in cfg.TIME_WINDOWS.get(country, []):
        pre_start = (pd.to_datetime(window['start']) - pd.Timedelta(days=90)).strftime('%Y-%m-%d')
        pre_end = (pd.to_datetime(window['start']) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')

        print(f'\n=== City features for {country} (ref: {window["start"]}) ===')
        df_cities = get_city_features(client, country, window['start'])
        print(f'  {len(df_cities)} cities')

        for treatment_city in candidates:
            run += 1
            print(f'\n=== [{run}] {treatment_city} | {window["start"]}-{window["end"]} ===')

            controls = find_control_cities(df_cities, treatment_city, cfg.CITY_KNN_NEIGHBORS)
            if not controls:
                continue
            print(f'  KNN: {len(controls)} candidates')

            all_cities = [treatment_city] + controls
            df_orders = get_daily_orders(client, country, all_cities, pre_start, window['post_end'])
            df_pre = df_orders[df_orders['orderdate'] <= pre_end]
            filtered = apply_correlation_filter(df_pre, treatment_city, controls, cfg.CITY_CORRELATION_THRESHOLD)
            if not filtered:
                continue

            pre_period = [pre_start, pre_end]
            post_period = [window['start'], window['end']]

            # Total level
            result = run_causal_impact(df_orders, treatment_city, filtered, pre_period, post_period)
            if result:
                incr = result['actual_orders'] - result['predicted_orders']
                write_city_result(client, {
                    'technique': 'CITY_BSTS', 'country': country,
                    'window_start': window['start'], 'window_end': window['end'],
                    'seed': None, 'treatment_city': treatment_city,
                    'base_segment': 'total', 'n_units': 1,
                    'n_controls': len(filtered), 'exact_match_pct': None,
                    'pre_period_uplift': None,
                    'campaign_period_uplift': result['uplift'],
                    'post_period_uplift': None,
                    'campaign_incr_orders': incr, 'post_incr_orders': None,
                    'run_timestamp': pd.Timestamp.now(tz='Europe/Amsterdam')})
                print(f'  Total uplift: {result["uplift"]:+.4f}')

            # Per segment
            for segment in cfg.BASE_SEGMENTS:
                if segment == 'unknown': continue
                df_seg = get_daily_orders(client, country, all_cities, pre_start, window['post_end'], segment)
                if df_seg.empty: continue
                seg_filtered = apply_correlation_filter(
                    df_seg[df_seg['orderdate']<=pre_end], treatment_city, controls, cfg.CITY_CORRELATION_THRESHOLD)
                if not seg_filtered: continue
                seg_result = run_causal_impact(df_seg, treatment_city, seg_filtered, pre_period, post_period)
                if seg_result:
                    seg_incr = seg_result['actual_orders'] - seg_result['predicted_orders']
                    write_city_result(client, {
                        'technique': 'CITY_BSTS', 'country': country,
                        'window_start': window['start'], 'window_end': window['end'],
                        'seed': None, 'treatment_city': treatment_city,
                        'base_segment': segment, 'n_units': 1,
                        'n_controls': len(seg_filtered), 'exact_match_pct': None,
                        'pre_period_uplift': None,
                        'campaign_period_uplift': seg_result['uplift'],
                        'post_period_uplift': None,
                        'campaign_incr_orders': seg_incr, 'post_incr_orders': None,
                        'run_timestamp': pd.Timestamp.now(tz='Europe/Amsterdam')})

            del df_orders; gc.collect()

elapsed = (time.time() - start_time) / 60
print(f'\n{"="*50}')
print(f'City Lookalike A/A complete in {elapsed:.1f} minutes.')
print(f'Results written to {cfg.AA_RESULTS_TABLE}')

---
## 7. Quick Summary

In [ ]:
df_check = client.query(f"""
    SELECT treatment_city, base_segment, campaign_period_uplift
    FROM `{cfg.AA_RESULTS_TABLE}`
    WHERE technique = 'CITY_BSTS'
    ORDER BY treatment_city, base_segment
""").to_dataframe()
print('City Lookalike results:')
df_check